# cube fitting example


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy import constants as const
from astropy import units as u
from astropy.io import fits
from spectral_cube import SpectralCube

from simplefit import fit_cube, fit_spectrum, plot_fit

%matplotlib inline


In [ ]:
INPUT_FILE = "./example_data/HNCO_cube_20arcsec_m100_p100kms_narrowlines.fits"

In [ ]:
cube = SpectralCube.read(INPUT_FILE)
cube = cube.with_spectral_unit(u.km / u.s, velocity_convention="radio")

## Fit A Small Mask With SSA-Guided Guesses

This fits a small central patch so the notebook runs quickly. `ssa_size=3` builds mask-aware 3x3 spectral averaging areas, auto-detects components once per SSA, then uses those guesses for the pixels inside each SSA.


In [ ]:
mask = np.zeros(cube.shape[1:], dtype=bool)
cy, cx = np.array(cube.shape[1:]) // 2
mask[cy - 2 : cy + 3, cx - 2 : cx + 3] = True

cube_fit = fit_cube(cube, n_jobs=2, mask=mask, ssa_size=3, max_components=None)
cube_fit.ssa_table


In [ ]:
print(f"Masked pixels: {mask.sum()}")
print(f"SSA auto-detection fits: {len(cube_fit.ssa_table)}")
print(f"Successful pixel fits: {cube_fit.success[mask].sum()}")
print(f"Total fitted components: {len(cube_fit.component_table)}")
print(f"Maximum components in one pixel: {cube_fit.n_components.max()}")


In [ ]:
cube_fit.component_table[:10]


## SSA-Guided Cube Fit Summary Maps


In [ ]:
first_center = cube_fit.component_map("center", component=1)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
images = [
    axes[0].imshow(cube_fit.ssa_labels, origin="lower", cmap="tab20"),
    axes[1].imshow(cube_fit.n_components, origin="lower", cmap="viridis"),
    axes[2].imshow(first_center, origin="lower", cmap="magma"),
]
axes[0].set_title("SSA labels")
axes[1].set_title("Number of components")
axes[2].set_title("Component 1 center")
for ax, image in zip(axes, images):
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
    fig.colorbar(image, ax=ax, shrink=0.85)
plt.show()


In [ ]:
component_table = cube_fit.component_table
valid_components = component_table[component_table['success'] & (component_table['fwhm'] > 0)]
min_fwhm = np.full(cube_fit.n_components.shape, np.nan, dtype=float)
for row in valid_components:
    y = int(row['y'])
    x = int(row['x'])
    value = float(row['fwhm'])
    if np.isnan(min_fwhm[y, x]):
        min_fwhm[y, x] = value
    else:
        min_fwhm[y, x] = min(min_fwhm[y, x], value)

fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
image = ax.imshow(min_fwhm, origin='lower', cmap='magma')
ax.contour(cube_fit.ssa_labels > 0, levels=[0.5], colors='white', linewidths=0.8, origin='lower')
ax.set_title('Minimum FWHM per fitted pixel')
ax.set_xlabel('x pixel')
ax.set_ylabel('y pixel')
fig.colorbar(image, ax=ax, shrink=0.85, label='FWHM')
plt.show()


## Save Sparse Component Table


In [ ]:
OUTPUT_TABLE = Path("/private/tmp/cube_fit_components.csv")
cube_fit.write_table(OUTPUT_TABLE)
OUTPUT_TABLE